<h1>ASSIGNMENT 4 INSTRUCTIONS</h1>



Gradio is an open-source Python package that allows you to quickly build a demo or web application for your machine learning model, API, or any arbitrary Python function.

**This assignment accomplishes the following objectives:**
1. In Assignment 3, we used three key design principles (tool use, reflection, and planning) to create an agentic AI workflow for blogging: Planner(Groq) -> Researcher(OpenAI + web_search tool) -> Writer(OpenAI) -> Editor(Groq) -> Reviser(OpenAI).
2. In this assignment, we will learn how to use Gradio to convert our blogging workflow to a web application.
3. We will also make the output from each step in the workflow editable by the user (before the workflow is allowed to proceed to the next step) - thereby demonstrating a human-in-the-loop agenticAI workflow.
4. Finally, we will reflect on the experience of being the "human in the loop" to understand where human intervention adds the most value in an agentic workflow.

**In order to complete the assignment:**
1. Section 1: Assignment setup. Just read through to understand what's already available.
2. Section 2: Blogging AgenticAI Workflow functions. Just read through and note how we've wrapped the blogging agenticAI workflow roles from Assignment 3 in functions.
3. Section 3: The Human-in-the-Loop Blogging AgenticAI Workflow Gradio app. Just read through and note how we're calling the functions defined in Section 3 to create a Human-in-the-Loop Gradio app for the agenticAI workflow.
4. Section 4: Reflect on your Human-in-the-Loop experience. Answer Q1 through Q7 inline.

**To submit:**
1. Rename the assignment as Assignment4-\<asurite\>
2. Runtime -> Restart session and run all
3. Ctrl-s
4. File -> Download .ipynb
5. Submit on Canvas

# SECTION 1: ASSIGNMENT SETUP (JUST READ THROUGH)

### We'll first import the required libraries.

In [1]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# install the orchestration layer (aisuite) plus the provider SDKs (OpenAI, Groq)
# in Colab, this is typically needed once per new runtime
!pip install aisuite openai groq

### We'll now setup the API KEYS and Clients.

In [2]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

import os
import aisuite
from openai import OpenAI
from groq import Groq

# load API KEYS (Google Colab + local env fallback)
try:
  # read the API KEYs from Colab Secrets and expose it as an env variable
  from google.colab import userdata
  os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
  os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
  # read the API KEYs from local env and expose it as an env variable
  os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY")
  os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY")

# create the clients
openai_client = OpenAI()
groq_client = Groq()
aisuite_client = aisuite.Client()

# select the models you want to use
OPENAI_MODEL= "gpt-4.1-mini"
AISUITE_OPENAI_MODEL = "openai:"+OPENAI_MODEL
GROQ_MODEL = "llama-3.3-70b-versatile"
AISUITE_GROQ_MODEL = "groq:"+GROQ_MODEL

### The reponses we get back from LLMs aren't always well-formatted. <br> We'll use a custom *show* utility function to print LLM responses.

In [3]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# a nifty utility function to cleanly diplay your chat outputs
from IPython.display import display, HTML, Markdown

def show(title, text):
    display(HTML(f"""
     <h4>{title}</h4>
     <div style="white-space: pre-wrap; padding-left: 24px;">{text}</div>"""))

### Recall from Assignment 1 that an LLM only has direct access to the information it was trained on and is unaware of events that occurred after its knowledge cutoff date. To address this limitation, we will provide access to a *web_search* tool that the LLM can call when it needs to retrieve current information. Since the OpenAI API provides a built-in web search capability, we will use it directly rather than implementing a custom solution.

In [4]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

import json

# use OpenAI's built-in web_search tool to perform web_search
def openai_web_search(query: str):
  """
    Perform a web search using OpenAI's built-in web_search tool
    and return raw results.
  """
  resp = openai_client.responses.create(
      model=OPENAI_MODEL,
      input=query,
      tools=[{"type": "web_search"}],
  )
  return json.loads(resp.model_dump_json())

### The *run_openai_query* function is similar to what we used in Assignment 1, with two key changes. First, it can now accept a list of tools and invoke them as needed. Second, instead of printing the LMM response and usage statistics inline, it returns both the model's response and relevant query details (including usage statistics and any tools that were called).

In [5]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# used for wall-clock timing for request
import time

# run the system/user prompt using OpenAI API, invoke tools as needed;
# return response and query details (usage stats and tools invoked)
def run_openai_query(system_prompt, user_prompt, tools=[]):
  """
    Execute the query, print the prompt, the response, and detailed usage stats.
    Supports tools.
  """

  # start the time before the API call
  start_time = time.time()

  # make chat-completions request thro aisuite, routed to OpenAI-backed model
  completion = aisuite_client.chat.completions.create(
      model=AISUITE_OPENAI_MODEL, # OpenAI-backed model
      messages=[
          {"role": "system", "content": system_prompt}, # system sets global behavior/instructions
           {"role": "user", "content": user_prompt} # user contains the actual task/prompt
      ],
      tools=tools,
      max_turns=3,
  )

  # compute end-to-end wall-clock time for API request
  elapsed_time = time.time() - start_time

  # response
  response = completion.choices[0].message.content.strip()

  # usage stats
  query_details = {}
  query_details["prompt_tokens"]=completion.usage.prompt_tokens
  query_details["completion_tokens"]=completion.usage.completion_tokens
  query_details["total_tokens"]=completion.usage.total_tokens
  if hasattr(completion.usage, 'total_time'):
    query_details["total_time"]=f"{completion.usage.total_time:.2f}s"
  else:
    query_details["total_time"]=f"{elapsed_time:.2f}s"

  # tools invoked
  tools_invoked=[]
  messages = completion.choices[0].intermediate_messages
  for m in messages or []:
    if hasattr(m, "tool_calls") and m.tool_calls:
      for t in m.tool_calls:
        tools_invoked.append(t.function.name)
  query_details["tools_invoked"]=tools_invoked

  return response, query_details


### The *run_groq_query* is similar to the *run_openai_query*, with one important difference: it does not accept/invoke any tools.

In [6]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# used for wall-clock timing for request
import time

# run the system/user prompt using OpenAI API; DOES NOT support tools;
# return response and query details (usage stats and tools invoked)
def run_groq_query(system_prompt, user_prompt):
  """
    Execute the query, print the prompt, the response, and detailed usage stats.
    Does not support tools.
  """

  # start the time before the API call
  start_time = time.time()

  # make chat-completions request thro aisuite, routed to Groq-backed model
  completion = aisuite_client.chat.completions.create(
      model=AISUITE_GROQ_MODEL, # Groq-backed model
      messages=[
          {"role": "system", "content": system_prompt}, # system sets global behavior/instructions
           {"role": "user", "content": user_prompt} # user contains the actual task/prompt
      ],
      max_turns=1,
  )

  # compute end-to-end wall-clock time for API request
  elapsed_time = time.time() - start_time

  # response
  response = completion.choices[0].message.content.strip()

  # usage stats
  query_details = {}
  query_details["prompt_tokens"]=completion.usage.prompt_tokens
  query_details["completion_tokens"]=completion.usage.completion_tokens
  query_details["total_tokens"]=completion.usage.total_tokens
  if hasattr(completion.usage, 'total_time'):
    query_details["total_time"]=f"{completion.usage.total_time:.2f}s"
  else:
    query_details["total_time"]=f"{elapsed_time:.2f}s"

  # tools invoked
  tools_invoked=[]
  messages = completion.choices[0].intermediate_messages
  for m in messages or []:
    if hasattr(m, "tool_calls") and m.tool_calls:
      for t in m.tool_calls:
        tools_invoked.append(t.function.name)
  query_details["tools_invoked"]=tools_invoked

  return response, query_details

# SECTION 2: Wraps the Blogging AgenticAI Workflow Roles in Functions (JUST READ THROUGH)

### AgenticAI Workflow

In this section, the Blogging AgenticAI Workflow roles have been wrapped in functions for you. Read through each function to understand the pattern before proceeding.

- def **planner** (topic: str) -> return plan
- def **researcher** (plan: str) -> return research
- def **writer** (plan: str, research: str) -> return draft
- def **editor** (plan: str, research: str, draft: str) -> return feedback
- def **reviser** (draft: str, feedback: str) -> return blog

### [No Coding: Planner Function]

The **planner** function is provided below:

def planner (topic: str) -> return plan

In [7]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# the planner takes a topic and returns a plan
def planner (topic: str):

  # planner system prompt
  planner_system_prompt = (
    f"""
    You are a planning agent for writing a blog post.
    Your job is to define structure and requirements, not to write prose.
    Be clear, concise, and structured.
    """
  )

  # planner user prompt
  planner_user_prompt = (
    f"""
    Blog topic:{topic}
    Create:
      1. A working blog title
      2. The target audience (1 sentence)
      3. The goal of the blog (1 sentence)
      4. A clear outline with 5-7 section headings
      5. A short research checklist of facts, statistics,
        or definitions that should be verified before writing
    Do not write the blog content.
    """
  )

  # use Groq (no tools) to create a plan
  plan, query_details = run_groq_query(planner_system_prompt, planner_user_prompt)

  # return the plan
  return plan

### [No Coding: Researcher Function]

The **researcher** function is provided below:

def researcher (plan: str) -> return research

In [8]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# the researcher takes a plan and returns the research
def researcher (plan: str):

  # researcher system prompt
  researcher_system_prompt = (
    f"""
    You are a research assistant.
    Use web search to find accurate, up-to-date information.
    Do not write blog prose.
    Return factual notes with sources.
    """
  )

  # research user prompt
  researcher_user_prompt = (
    f"""
    Blog plan (outline + checklist):{plan}
    For each outline section:
      1. Provide 3-5 factual bullet points
      2. Include definitions or statistics where relevant
      3. Include a source for each section
    Return concise research notes only.
    """
  )

  # use OpenAI (with web_search) to do the research
  research, query_details = run_openai_query(researcher_system_prompt, researcher_user_prompt, tools=[openai_web_search])

  # return the research
  return research

### [No Coding: Writer Function]

The **writer** function is provided below:

def writer (plan: str, research: str) -> return draft

In [9]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# the writer takes the plan and research and returns a draft
def writer (plan: str, research: str):

  # writer system prompt
  writer_system_prompt = (
    f"""
    You are a blog writer.
    Write clearly and engagingly for the target audience.
    Focus on structure, flow, and clarity.
    Do not critique your writing.
    """
  )

  # writer user prompt
  writer_user_prompt = (
    f"""
    Audience, goal, and outline from planner:{plan}
    Research and sources from researcher:{research}
    Write a complete one-page blog post.
    Use the audience, goal, and outline accurately.
    Use the research and sources accurately.
    """
  )

  # use OpenAI (no tools) to write a draft
  draft, query_details = run_openai_query(writer_system_prompt, writer_user_prompt, tools=[])

  # return the draft
  return draft

### [No Coding: Editor Function]

The **editor** function is provided below:

def editor (plan: str, research: str, draft: str) -> return feedback

In [10]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# the editor takes the plan, research, and draft, and returns feedback
def editor (plan: str, research: str, draft: str):

  # editor system prompt
  editor_system_prompt = (
    f"""
    You are a critical editor.
    Your job is to evaluate and suggest improvements, not to rewrite the content.
    Be specific and constructive.
    """
  )

  # editor user prompt
  editor_user_prompt = (
    f"""
    Original audience, goal, and outline from planner:{plan}
    Research and sources from researcher:{research}
    Draft from writer:{draft}
    Evaluate on: clarity, structure/flow, accuracy (given the research), and tone.
    Provide:
      1. A 1-5 score for each category
      2. Specific, actionable improvement suggestions
    Do not rewrite the blog.
    """
  )

  # use Groq (no tools) provide feedback on the draft
  feedback, query_details = run_groq_query(editor_system_prompt, editor_user_prompt)

  # return feedback
  return feedback

### [No Coding: Reviser Function]

The **reviser** function is provided below:

def reviser (draft: str, feedback: str) -> return blog

In [11]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# the reviser takes the draft and feedback and returns a blog
def reviser (draft: str, feedback: str):

  # reviser system prompt
  reviser_system_prompt = (
    f"""
    You are a blog writer revising a draft based on editorial feedback.
    Improve clarity, structure, and accuracy.
    Preserve the original intent and outline unless told otherwise.
    """
  )

  # reviser user prompt
  reviser_user_prompt = (
    f"""
    Original blog draft:{draft}
    Editor feedback:{feedback}
    Revise the blog post using the feedback.
    Return the full revised one-page blog post.
    """
  )

  # use OpenAI (no tools) creat the blog
  blog, query_details = run_openai_query(reviser_system_prompt, reviser_user_prompt, tools=[])

  # return the blog
  return blog

# SECTION 3: Creates a Human-in-the-Loop Blogging AgenticAI Workflow using Gradio

### [No Coding: Run the Human-in-the-Loop Gradio App]

All Gradio code has been provided for you. Just run the cell below to launch the app.

Use the public URL that Gradio provides in the cell output for a cleaner view. Then work through the full blogging workflow on any topic, for instance: **"The Impact of Artificial Intelligence on the Future of Work"**

At each editable stage (Plan, Research, Draft, Feedback), read the output carefully and make at least one meaningful edit before proceeding to the next step. Keep notes on what you changed and why — you will answer questions about this in Section 4.

In [12]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

import gradio as gr

# build a human-in-the-loop agenticAI workflow UI
with gr.Blocks(title="Blog AgenticAI Workflow") as app:

    # Textbox to provide the topic input
    topic = gr.Textbox(label="Topic", placeholder="e.g. The Impact of Artificial Intelligence on the Future of Work")

    # Button that triggers the planner when clicked
    submit_topic_btn = gr.Button("Submit Topic → Run Planner")


    # Textbox to display the (editable) planner output
    plan = gr.Textbox(label="Plan from Planner (editable)", lines=10)

    # When the submit topic button is clicked, call planner(topic) and fill `plan`
    # The plan is editable by user
    submit_topic_btn.click(fn=planner, inputs=[topic], outputs=[plan])

    # Button that triggers the researcher when clicked
    submit_plan_btn = gr.Button("Submit Plan → Run Researcher")


    # Textbox to display the (editable) researcher output
    research = gr.Textbox(label="Research from Researcher (editable)", lines=10)

    # When the submit plan button is clicked, call researcher(plan) and fill `research`
    # The research is editable by user
    submit_plan_btn.click(fn=researcher, inputs=[plan], outputs=[research])

    # Button that triggers the writer when clicked
    submit_research_btn = gr.Button("Submit Research → Run Writer")


    # Textbox to display the (editable) writer output
    draft = gr.Textbox(label="Draft from Writer (editable)", lines=10)

    # When the submit research button is clicked, call writer(plan, research) and fill `draft`
    # The draft is editable by user
    submit_research_btn.click(fn=writer, inputs=[plan, research], outputs=[draft])

    # Button that triggers the editor when clicked
    submit_draft_btn = gr.Button("Submit Draft → Run Editor")


    # Textbox to display the (editable) editor feedback
    feedback = gr.Textbox(label="Feedback from Editor (editable)", lines=10)

    # When the submit draft button is clicked, call editor(plan, research, draft) and fill `feedback`
    # The feedback is editable by user
    submit_draft_btn.click(fn=editor, inputs=[plan, research, draft], outputs=[feedback])

    # Button that triggers the reviser when clicked
    submit_feedback_btn = gr.Button("Submit Feedback → Run Reviser")


    # Textbox to display the final output blog
    blog = gr.Textbox(label="Final Blog from Reviser", lines=15)

    # When the submit feedback button is clicked, call reviser(draft, feedback) and fill `blog`
    submit_feedback_btn.click(fn=reviser, inputs=[draft, feedback], outputs=[blog])

# Start the Gradio web application
app.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://65bcecbf78bef334e8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# SECTION 4: Reflect on Your Human-in-the-Loop Experience (ANSWER Q1 THROUGH Q7)

### Now that you've run the full workflow and edited the outputs at each stage, answer the six questions below.

Answer each question inline inside the code cell that follows, as a short comment (1-3 sentences). There is no single right answer — what matters is that your answer reflects genuine engagement with the workflow you just ran.

### Q1: What are some topics you tested the workflow with, and why?

Think about why you picked the topics you did, what topics did the workflow do well on, and why you think it did well on those specific topics.

In [13]:
##############################################################
#  Q1) What are some topics you tested the workflow with, and why?
# # Q1:
# I tested the workflow using a topic on customer retention for a martial arts academy since it directly relates to my business. I chose this so I could evaluate
# if the output was actually useful and realistic. The workflow did okay, but struggled at times because the topic is more niche and doesn’t have as much general data
# compared to broader business topics.
#
##############################################################

### Q2: What did you edit in the Plan stage, and why?

Think about what you changed (a section heading, the target audience, the outline structure, etc.) and why it mattered for the steps downstream.

In [14]:
##############################################################
#  Q2) What did you edit in the Plan stage, and why?
# In the Plan stage, I adjusted the outline to be more practical and focused on real retention strategies instead of general concepts. I also made sure the target audience was clearly
# martial arts academy owners. This mattered because it helped guide the research and draft stages to be more relevant instead of generic.
#
##############################################################

### Q3: What did you edit in the Research stage, and why?

Think about what you changed (a statistic, a source, a missing topic, etc.) and why it mattered for the draft downstream.

In [15]:
##############################################################
#  Q3) What did you edit in the Research stage, and why?
# In the Research stage, I added more relevant examples and adjusted some of the points to better fit martial arts instead of general fitness.
# I also removed a few weak or vague supporting details. This mattered because the draft relies on the research, and better input led to a more realistic and useful final blog.
#
##############################################################

### Q4: What did you edit in the Draft stage, and why?

Think about what you changed (tone, structure, accuracy, etc.) and why a human was better positioned than the LLM to catch it.

In [16]:
##############################################################
#  Q4) What did you edit in the Draft stage, and why?
# In the Draft stage, I adjusted the tone to sound more practical and less generic, and cleaned up a few sections for clarity. I also corrected some parts that didn’t
# fully match how a martial arts academy actually operates. A human was better positioned here because I understand the real-world context and what would actually make sense to the audience.
#
##############################################################

### Q5: What did you edit in the Feedback stage, and why?

Think about what you changed (prioritized certain suggestions, removed conflicting ones, etc.) and how that shaped the final blog.

In [17]:
##############################################################
#  Q5) What did you edit in the Feedback stage, and why?
# In the Feedback stage, I focused on keeping the suggestions that improved clarity and removed ones that were repetitive or didn’t add much value. Some feedback was too generic, so
# I filtered it down to what actually strengthened the blog. This helped shape a cleaner final version instead of overloading it with unnecessary changes.
#
##############################################################

### Q6: At which stage did human intervention feel most valuable, and why?

Consider which stage had the highest potential for errors to compound downstream if left uncorrected.

In [18]:
##############################################################
#  Q6) At which stage did human intervention feel most valuable, and why?
# Human intervention felt most valuable in the Plan stage because it sets the direction for everything that follows. If the plan is too generic or off-target, the research and draft
# will also be weak. Fixing it early prevents errors from carrying through the entire workflow.
#
##############################################################

### Q7: What does this tell you about where humans add the most value in agentic workflows generally?

Think beyond blogging — what general principle can you draw about when and where to put humans in the loop in any agentic AI workflow?

In [19]:
##############################################################
#  Q7) What does this tell you about where humans add the most
#      value in agentic workflows generally?
# This shows that humans add the most value at the early stages where direction and accuracy are set. If those steps are wrong, everything downstream gets worse. In general, humans are most
# useful for guiding, validating, and correcting key decisions rather than just generating content.
#
##############################################################